# Audio a Texto + Clasificación de Emergencias

**Pipeline completo:**
1. El usuario graba un audio describiendo una emergencia
2. Whisper (modelo `small`) transcribe el audio a texto
3. Llama 3.3 70B via Groq analiza el texto y clasifica la gravedad

**Requisitos previos:**
- Python 3.9+
- ffmpeg instalado en el sistema ([instrucciones](https://ffmpeg.org/download.html))
- API key **gratuita** de Groq — sin tarjeta de crédito:
  1. Ve a [console.groq.com](https://console.groq.com)
  2. Regístrate con Google o email
  3. Menú izquierdo → **API Keys** → **Create API Key**
  4. Copia la key y pégala abajo en `GROQ_API_KEY`

## 0. Instalación de dependencias
Ejecuta esta celda solo la primera vez.

In [ ]:
!pip install openai-whisper groq

## 1. Configuración

In [ ]:
import whisper
import warnings
import json
import os
import time
from groq import Groq

warnings.filterwarnings("ignore")

# =============================================
# CONFIGURACIÓN - Modifica estos valores
# =============================================

# Tu API key gratuita de Groq (console.groq.com → API Keys → Create API Key)
GROQ_API_KEY = ""

# Modelo de Whisper para transcripción:
# 'tiny' (39MB) | 'base' (74MB) | 'small' (461MB) | 'medium' (1.5GB)
# Recomendado 'small': buen equilibrio precisión/velocidad para español
WHISPER_MODEL = "small"

# Modelo de Groq para clasificación.
# llama-3.3-70b-versatile: el más potente del tier gratuito
# llama-3.1-8b-instant: más ligero y con más cuota diaria (14.400 req/día)
GROQ_MODEL = "llama-3.3-70b-versatile"

# Idioma del audio
IDIOMA = "es"

print("✓ Configuración cargada.")

## 2. Carga del modelo Whisper
La primera ejecución descargará el modelo (~461 MB para `small`). Las siguientes será instantáneo.

In [ ]:
print(f"Cargando modelo Whisper '{WHISPER_MODEL}'...")
inicio = time.time()

modelo_whisper = whisper.load_model(WHISPER_MODEL)

print(f"✓ Modelo cargado en {time.time() - inicio:.1f} segundos.")

## 3. Funciones del pipeline

In [ ]:
def transcribir_audio(ruta_audio: str) -> dict:
    """
    Transcribe un archivo de audio a texto usando Whisper.

    Args:
        ruta_audio: Ruta al archivo de audio (.mp3, .wav, .m4a, etc.)

    Returns:
        dict con 'exito' (bool) y 'texto' (str)
    """
    if not os.path.isfile(ruta_audio):
        return {"exito": False, "texto": f"Archivo no encontrado: '{ruta_audio}'"}

    try:
        resultado = modelo_whisper.transcribe(
            ruta_audio,
            language=IDIOMA,
            fp16=False  # Compatibilidad con CPU
        )
        return {"exito": True, "texto": resultado["text"].strip()}

    except Exception as e:
        return {"exito": False, "texto": f"Error en transcripción: {e}"}


def clasificar_emergencia(texto: str) -> dict:
    """
    Usa Llama 3.3 70B via Groq para clasificar la gravedad de una emergencia.

    Args:
        texto: Texto transcrito del audio de emergencia

    Returns:
        dict con 'nivel', 'justificacion', 'accion_recomendada' y 'servicios'
    """
    cliente = Groq(api_key=GROQ_API_KEY)

    prompt_sistema = """Eres un operador experto de un centro de emergencias 112.
Analiza mensajes transcritos de llamadas de emergencia y clasifícalos.

Responde ÚNICAMENTE con un JSON válido, sin bloques de código markdown, con esta estructura:
{
    "nivel": "LEVE" | "MEDIA" | "GRAVE",
    "justificacion": "Explicación breve de por qué has elegido ese nivel",
    "accion_recomendada": "Qué acción o servicio se recomienda activar",
    "servicios": ["lista de servicios necesarios, ej: ambulancia, bomberos, policia"]
}

Criterios de clasificación:
- LEVE: Sin riesgo vital. Averías, ruidos, pequeños accidentes sin heridos, consultas.
- MEDIA: Riesgo potencial sin peligro inmediato. Accidentes con heridos leves, incendios
  pequeños, robos sin violencia, personas perdidas.
- GRAVE: Riesgo vital inminente. Personas inconscientes o con heridas graves, incendios
  descontrolados, violencia activa, derrumbes, atrapados."""

    try:
        respuesta = cliente.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {"role": "system", "content": prompt_sistema},
                {"role": "user", "content": f"Clasifica esta transcripción:\n\n\"{texto}\""}
            ],
            temperature=0,  # Respuestas deterministas: mejor para clasificación
            max_tokens=400
        )

        texto_respuesta = respuesta.choices[0].message.content.strip()
        # Limpiar posibles bloques de código markdown por si acaso
        texto_respuesta = texto_respuesta.replace("```json", "").replace("```", "").strip()
        return json.loads(texto_respuesta)

    except json.JSONDecodeError:
        return {
            "nivel": "ERROR",
            "justificacion": f"Respuesta no parseable: {texto_respuesta}",
            "accion_recomendada": "Revisión manual necesaria",
            "servicios": []
        }
    except Exception as e:
        return {
            "nivel": "ERROR",
            "justificacion": f"Error al clasificar: {e}",
            "accion_recomendada": "Revisión manual necesaria",
            "servicios": []
        }


def procesar_emergencia(ruta_audio: str) -> dict:
    """
    Pipeline completo: audio → transcripción → clasificación → resultado.
    """
    print(f"Procesando: {ruta_audio}")
    print("-" * 52)

    # Paso 1: Transcripción
    print("[1/2] Transcribiendo audio con Whisper...")
    transcripcion = transcribir_audio(ruta_audio)

    if not transcripcion["exito"]:
        print(f"✗ Error: {transcripcion['texto']}")
        return {"error": transcripcion["texto"]}

    print(f"✓ Texto: \"{transcripcion['texto']}\"\n")

    # Paso 2: Clasificación
    print(f"[2/2] Clasificando con {GROQ_MODEL} via Groq...")
    clasificacion = clasificar_emergencia(transcripcion["texto"])

    # Mostrar resultado
    nivel = clasificacion.get("nivel", "DESCONOCIDO")
    indicador = {"LEVE": "🟢", "MEDIA": "🟡", "GRAVE": "🔴"}.get(nivel, "⚪")

    print()
    print("=" * 52)
    print("           RESULTADO DE LA EMERGENCIA")
    print("=" * 52)
    print(f"  {indicador}  Nivel:              {nivel}")
    print(f"\n  Transcripción:")
    print(f"  \"{transcripcion['texto']}\"")
    print(f"\n  Justificación:")
    print(f"  {clasificacion.get('justificacion', 'N/A')}")
    print(f"\n  Acción recomendada:")
    print(f"  {clasificacion.get('accion_recomendada', 'N/A')}")
    print(f"\n  Servicios a activar:")
    for s in clasificacion.get("servicios", []):
        print(f"    - {s}")
    print("\n" + "=" * 52)

    return {
        "audio": ruta_audio,
        "transcripcion": transcripcion["texto"],
        "clasificacion": clasificacion
    }


print("✓ Funciones cargadas.")

## 4. Ejecución
Cambia `mi_audio` por la ruta de tu archivo de audio y ejecuta la celda.

In [ ]:
# Cambia esto por el nombre o ruta de tu archivo de audio
mi_audio = "Grabacion.m4a"

resultado = procesar_emergencia(mi_audio)

## 5. (Opcional) Probar solo la clasificación con texto
Útil para verificar que tu API key de Groq funciona sin necesitar un audio.

In [ ]:
textos_prueba = [
    "Hola, quería avisar de que hay un bache bastante grande en la calle Mayor.",
    "Ha habido un accidente en la A3, hay dos coches implicados y parece que hay un herido leve.",
    "¡Por favor, ayuda! Hay un incendio en el edificio de al lado, hay gente atrapada en el tercer piso."
]

for i, texto in enumerate(textos_prueba, 1):
    print(f"\n{'='*52}")
    print(f"PRUEBA {i}: \"{texto}\"")
    print("="*52)

    clasif = clasificar_emergencia(texto)
    nivel = clasif.get("nivel", "?")
    indicador = {"LEVE": "🟢", "MEDIA": "🟡", "GRAVE": "🔴"}.get(nivel, "⚪")

    print(f"  {indicador} Nivel:        {nivel}")
    print(f"  Justificación: {clasif.get('justificacion', 'N/A')}")
    print(f"  Servicios:     {', '.join(clasif.get('servicios', []))}")